# 01 — Exploratory Data Analysis
## Phishing Email Detection with Explainable AI

**Dataset:** 82,486 emails (42,891 phishing / 39,595 legitimate)  
**Goal:** Understand the structure of the data, class distribution, text characteristics, and key linguistic patterns before building the detection model.

---

In [ ]:
# ── Connect Google Drive ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

PHISHING_CSV = '/content/drive/MyDrive/phishing_email.csv'
KZ_CSV       = '/content/drive/MyDrive/kazakhstan_urls.csv'
print("Drive connected. Files ready.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Load Dataset

In [ ]:
df = pd.read_csv(PHISHING_CSV)
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
print('=== Basic Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())

## 2. Class Distribution

The dataset contains two classes:
- **0** — Legitimate email
- **1** — Phishing email

In [ ]:
counts = df['label'].value_counts()
labels_map = {0: 'Legitimate', 1: 'Phishing'}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
colors = ['#2ecc71', '#e74c3c']
axes[0].bar([labels_map[i] for i in counts.index], counts.values, color=colors, edgecolor='white', linewidth=1.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 300, f'{v:,}', ha='center', fontweight='bold', fontsize=12)
axes[0].set_title('Class Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Emails')
axes[0].set_ylim(0, max(counts.values) * 1.12)

# Pie chart
axes[1].pie(counts.values, labels=[labels_map[i] for i in counts.index],
            autopct='%1.1f%%', colors=colors, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Distribution (%)', fontsize=14, fontweight='bold')

plt.suptitle('Figure 3.1. Dataset Class Distribution', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('fig_3_1_class_distribution.png', bbox_inches='tight')
plt.show()

print(f'Legitimate: {counts[0]:,} ({counts[0]/len(df)*100:.1f}%)')
print(f'Phishing:   {counts[1]:,} ({counts[1]/len(df)*100:.1f}%)')
print(f'Class imbalance ratio: {counts[1]/counts[0]:.2f}')

**Observation:** The dataset is nearly balanced (52.0% phishing vs 48.0% legitimate), which reduces the risk of model bias toward the majority class. This is favourable for training without requiring aggressive resampling strategies.

## 3. Text Length Analysis

In [ ]:
df['text_length'] = df['text_combined'].str.len()
df['word_count'] = df['text_combined'].str.split().str.len()

print('=== Text Length Statistics ===')
print(df.groupby('label')[['text_length', 'word_count']].describe().round(1))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

label_names = {0: 'Legitimate', 1: 'Phishing'}
pal = {0: '#2ecc71', 1: '#e74c3c'}

for ax, col, title in zip(axes[0], ['text_length', 'word_count'],
                          ['Character Length', 'Word Count']):
    for lbl in [0, 1]:
        subset = df[df['label'] == lbl][col].clip(upper=df[col].quantile(0.99))
        ax.hist(subset, bins=60, alpha=0.6, color=pal[lbl], label=label_names[lbl], edgecolor='none')
    ax.set_title(f'Distribution of {title}', fontweight='bold')
    ax.set_xlabel(title)
    ax.set_ylabel('Frequency')
    ax.legend()

# Box plots
for ax, col, title in zip(axes[1], ['text_length', 'word_count'],
                          ['Character Length', 'Word Count']):
    plot_data = [df[df['label'] == lbl][col].clip(upper=df[col].quantile(0.99)).values for lbl in [0, 1]]
    bp = ax.boxplot(plot_data, labels=['Legitimate', 'Phishing'], patch_artist=True)
    for patch, color in zip(bp['boxes'], ['#2ecc71', '#e74c3c']):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(f'Box Plot: {title} by Class', fontweight='bold')
    ax.set_ylabel(title)

plt.suptitle('Figure 3.2. Text Length Distribution by Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_3_2_text_length.png', bbox_inches='tight')
plt.show()

## 4. Keyword & Pattern Analysis

In [ ]:
# Phishing-related keywords presence
phishing_keywords = [
    'click', 'verify', 'account', 'password', 'urgent', 'free',
    'winner', 'prize', 'bank', 'suspend', 'confirm', 'login',
    'update', 'security', 'limited', 'offer', 'deal', 'http'
]

keyword_rates = {}
for kw in phishing_keywords:
    phish_rate = df[df['label']==1]['text_combined'].str.contains(kw, case=False).mean()
    legit_rate = df[df['label']==0]['text_combined'].str.contains(kw, case=False).mean()
    keyword_rates[kw] = {'Phishing': phish_rate, 'Legitimate': legit_rate}

kw_df = pd.DataFrame(keyword_rates).T.sort_values('Phishing', ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
x = np.arange(len(kw_df))
w = 0.35
ax.barh(x - w/2, kw_df['Phishing'], w, label='Phishing', color='#e74c3c', alpha=0.85)
ax.barh(x + w/2, kw_df['Legitimate'], w, label='Legitimate', color='#2ecc71', alpha=0.85)
ax.set_yticks(x)
ax.set_yticklabels(kw_df.index, fontsize=11)
ax.set_xlabel('Proportion of Emails Containing Keyword')
ax.set_title('Figure 3.3. Keyword Presence Rate by Class', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.savefig('fig_3_3_keyword_rates.png', bbox_inches='tight')
plt.show()

## 5. URL and Structural Feature Presence

In [ ]:
df['has_url']          = df['text_combined'].str.contains(r'http[s]?://', regex=True).astype(int)
df['has_ip_url']       = df['text_combined'].str.contains(r'http[s]?://\d{1,3}\.\d{1,3}', regex=True).astype(int)
df['has_urgent_word']  = df['text_combined'].str.contains(r'\b(urgent|immediately|verify now|act now|limited time)\b',
                                                           case=False, regex=True).astype(int)
df['has_free']         = df['text_combined'].str.contains(r'\bfree\b', case=False, regex=True).astype(int)
df['has_click_here']   = df['text_combined'].str.contains(r'click here', case=False, regex=True).astype(int)
df['has_password']     = df['text_combined'].str.contains(r'\bpassword\b', case=False, regex=True).astype(int)
df['has_account']      = df['text_combined'].str.contains(r'\baccount\b', case=False, regex=True).astype(int)
df['has_winner']       = df['text_combined'].str.contains(r'\b(winner|won|congratulations)\b', case=False, regex=True).astype(int)

structural_features = ['has_url', 'has_ip_url', 'has_urgent_word', 'has_free',
                        'has_click_here', 'has_password', 'has_account', 'has_winner']

feat_rates = df.groupby('label')[structural_features].mean().T
feat_rates.columns = ['Legitimate', 'Phishing']
feat_rates = feat_rates.sort_values('Phishing', ascending=False)

print('=== Structural Feature Presence Rate ===')
print((feat_rates * 100).round(1).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
feat_rates.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'], alpha=0.85, edgecolor='white', linewidth=0.5)
ax.set_xticklabels([f.replace('has_', '').replace('_', ' ').title() for f in feat_rates.index],
                   rotation=30, ha='right')
ax.set_ylabel('Proportion of Emails')
ax.set_title('Figure 3.4. Structural Feature Presence by Class', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend(['Legitimate', 'Phishing'])
plt.tight_layout()
plt.savefig('fig_3_4_structural_features.png', bbox_inches='tight')
plt.show()

## 6. Most Frequent Words by Class

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

stop_words = ['the', 'a', 'an', 'is', 'it', 'in', 'on', 'at', 'to', 'for',
              'of', 'and', 'or', 'be', 'this', 'that', 'are', 'was', 'with',
              'i', 'you', 'we', 'he', 'she', 'they', 'from', 'by', 'as',
              'your', 'our', 'their', 'its', 'my', 'com', 'http', 'www']

def top_words(series, n=20):
    vec = CountVectorizer(max_features=5000, stop_words=stop_words, min_df=5)
    X = vec.fit_transform(series.fillna(''))
    counts = np.asarray(X.sum(axis=0)).flatten()
    vocab = vec.get_feature_names_out()
    top = sorted(zip(vocab, counts), key=lambda x: x[1], reverse=True)[:n]
    return pd.DataFrame(top, columns=['word', 'count'])

top_phish = top_words(df[df['label']==1]['text_combined'])
top_legit = top_words(df[df['label']==0]['text_combined'])

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, data, title, color in zip(
    axes,
    [top_phish, top_legit],
    ['Top 20 Words — Phishing', 'Top 20 Words — Legitimate'],
    ['#e74c3c', '#2ecc71']
):
    ax.barh(data['word'][::-1], data['count'][::-1], color=color, alpha=0.8)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xlabel('Frequency')

plt.suptitle('Figure 3.5. Most Frequent Words by Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_3_5_top_words.png', bbox_inches='tight')
plt.show()

## 7. EDA Summary

| Finding | Detail |
|---------|--------|
| Dataset size | 82,486 emails |
| Class balance | 52.0% phishing / 48.0% legitimate — nearly balanced |
| Missing values | None |
| URL presence | Significantly higher in phishing emails |
| Urgency language | ~2× more common in phishing |
| 'Free' / 'winner' | Strong phishing indicators |
| Text length | Phishing emails tend to be shorter on average |

**Conclusion:** The dataset contains clear textual and structural patterns that distinguish phishing from legitimate emails. These patterns will form the basis for interpretable feature engineering in the next notebook.